# ema-second-moment — worked example 2: Multi-Parameter v-Buffer: Comparing Two beta2 Values

> Worked example from [Delta Drills](https://delta-drills.vercel.app). Atom: `ema-second-moment`.

**This is a worked example — read it, run it, follow the reasoning.** It is study material, not a graded drill (no completion beacon). When the steps feel obvious, move to the faded version, then the full drill.

## Setup

In [ ]:
import numpy as np
import torch as t
from torch import Tensor
import einops
from einops import rearrange, reduce, repeat

t.manual_seed(0)
np.random.seed(0)

## Concept

The second-moment buffer `v` in Adam tracks a per-coordinate EMA of squared gradients. Higher `beta2` (closer to 1) makes the EMA adapt more slowly — old gradient history is discounted less aggressively. Lower `beta2` (e.g. 0.9) causes `v` to react quickly to gradient changes. Comparing the two values after several steps illustrates how `beta2` controls the 'memory' of the optimizer.

## Worked solution

We have two parameters with different gradient histories: param A has had consistently large gradients, param B has had small gradients. We run 10 steps with `beta2 = 0.9` and `beta2 = 0.999` and compare.

**Setup:** We keep two separate `v` tensors — `v_fast` (beta2=0.9) and `v_slow` (beta2=0.999) — both starting at zero. The gradients for the two parameters are fixed at 3.0 and 0.5 respectively.

**Why v_fast converges faster:** With `beta2 = 0.9`, the weight on the new `g^2` term is `0.1` — quite large. After 10 steps, `v_fast` is already very close to `g^2`. With `beta2 = 0.999`, the weight is only `0.001`, so `v_slow` is still far from the true `g^2` value after 10 steps.

**Practical implication:** A low `beta2` makes Adam's per-coordinate step size adapt quickly to gradient magnitude changes; a high `beta2` provides a smoother, more stable estimate at the cost of slower adaptation to gradient shifts.

In [ ]:
import torch as t

t.manual_seed(42)

beta2_fast = 0.9
beta2_slow = 0.999
grads = t.tensor([3.0, 0.5])  # two parameter gradients

v_fast = t.zeros(2)
v_slow = t.zeros(2)

for step in range(10):
    v_fast.copy_(beta2_fast * v_fast + (1 - beta2_fast) * grads.pow(2))
    v_slow.copy_(beta2_slow * v_slow + (1 - beta2_slow) * grads.pow(2))

g_sq = grads.pow(2)
print("g^2 (target):", g_sq.tolist())
print("v_fast after 10 steps (beta2=0.9):   ", [f"{x:.4f}" for x in v_fast.tolist()])
print("v_slow after 10 steps (beta2=0.999): ", [f"{x:.4f}" for x in v_slow.tolist()])

# v_fast should be much closer to g^2
err_fast = (v_fast - g_sq).abs()
err_slow = (v_slow - g_sq).abs()
assert (err_fast < err_slow).all(), "beta2=0.9 should converge faster than beta2=0.999"
print("Confirmed: lower beta2 converges faster to g^2.")